# Annotation -> Resized + Deduped Datasets

Reads a fresh `annotation.csv` and appends only genuinely-new rows to `landslide.csv`
and `non_landslide.csv`.

For every run:
1. read the fresh `annotation.csv`,
2. RESIZE every row to the fixed `TARGET_BOX_SIZE_M` (centroid-preserving),
3. DEDUPLICATE within the annotation file itself (IoU >= `DUP_IOU`, per label per incident),
4. COMPARE each deduped row against the existing `landslide.csv` / `non_landslide.csv`,
   skipping rows whose location already exists (IoU >= `DUP_IOU`),
5. APPEND only the surviving rows to the matching output (landslide vs not_landslide).

In [1]:
import os, math
import pandas as pd
import numpy as np

# ---------------- inputs ----------------
ANNOTATION_CSV = 'annotation.csv'
INCIDENT_CSV   = 'landslides_from_2018_to_2026.csv'

INCIDENT_ID_COL   = 'id'
INCIDENT_DATE_COL = 'incident_on'   # inventory dates are DD-MM-YYYY, e.g. 20-05-2026
INCIDENT_DATE_FMT = '%d-%m-%Y'

def load_inventory():
    inv = pd.read_csv(INCIDENT_CSV)
    inv[INCIDENT_ID_COL] = inv[INCIDENT_ID_COL].astype(str).str.strip()
    inv[INCIDENT_DATE_COL] = pd.to_datetime(
        inv[INCIDENT_DATE_COL], format=INCIDENT_DATE_FMT, errors='coerce', dayfirst=True)
    return inv

# ---------------- outputs ----------------
# Files hold ONLY resized + deduped rows; appended incrementally per run.
LANDSLIDE_CSV     = 'landslide.csv'
NON_LANDSLIDES_CSV = 'non_landslide.csv'

TARGET_BOX_SIZE_M = 1000.0   # fixed, centroid-preserving box size
DUP_IOU          = 0.50   # IoU threshold (within-file and against existing rows)

In [2]:
def iou(b1, b2):
    ix = max(0.0, min(b1['max_lon'], b2['max_lon']) - max(b1['min_lon'], b2['min_lon']))
    iy = max(0.0, min(b1['max_lat'], b2['max_lat']) - max(b1['min_lat'], b2['min_lat']))
    inter = ix * iy
    a1 = (b1['max_lon'] - b1['min_lon']) * (b1['max_lat'] - b1['min_lat'])
    a2 = (b2['max_lon'] - b2['min_lon']) * (b2['max_lat'] - b2['min_lat'])
    union = a1 + a2 - inter
    return inter / union if union > 0 else 0.0

def fixed_bbox_ann(row):
    lat_c = (row['min_lat'] + row['max_lat']) / 2.0
    lon_c = (row['min_lon'] + row['max_lon']) / 2.0
    deg_lat = TARGET_BOX_SIZE_M / 111000.0
    deg_lon = TARGET_BOX_SIZE_M / (111000.0 * math.cos(math.radians(lat_c)))
    return {
        'min_lon': lon_c - deg_lon / 2.0,
        'max_lon': lon_c + deg_lon / 2.0,
        'min_lat': lat_c - deg_lat / 2.0,
        'max_lat': lat_c + deg_lat / 2.0,
    }

In [3]:
# Load, validate, and resize the fresh annotation.csv.
ann = pd.read_csv(ANNOTATION_CSV)
for cc in ['incident', 'label']:
    ann[cc] = ann[cc].astype(str).str.strip()
ann['label'] = ann['label'].str.lower()
ann['candidate'] = ann['candidate'].astype(str).str.extract(r'(\d+)', expand=False)
for cc in ['candidate', 'min_lat', 'max_lat', 'min_lon', 'max_lon']:
    ann[cc] = pd.to_numeric(ann[cc], errors='coerce')
ann = ann.dropna(subset=['candidate', 'min_lat', 'max_lat', 'min_lon', 'max_lon']).copy()
ann['candidate'] = ann['candidate'].astype(int)
print('annotation rows (valid):', len(ann))

# Attach the incident date from the inventory (DD-MM-YYYY -> YYYY-MM-DD).
inventory = load_inventory()
inventory_dates = dict(zip(inventory[INCIDENT_ID_COL], inventory[INCIDENT_DATE_COL]))
ann['incident_date'] = ann['incident'].map(inventory_dates)
missing_date = ann['incident_date'].isna().sum()
print('incidents without a matching date (left empty):', missing_date)

# 1) resize to the fixed box.
rows = []
for _, r in ann.iterrows():
    fb = fixed_bbox_ann(r)
    inc_date = r['incident_date']
    rows.append({
        'incident': r['incident'], 'candidate': int(r['candidate']),
        'label': r['label'],
        'incident_date': pd.NaT if pd.isna(inc_date) else inc_date.strftime('%Y-%m-%d'),
        **fb,
    })
print('after resize:', len(rows))

annotation rows (valid): 200
incidents without a matching date (left empty): 185
after resize: 200


In [4]:
# 2) dedupe within the file: same label + IoU >= DUP_IOU -> keep one
#    overlap is checked across ALL candidate rows, regardless of incident.
from collections import defaultdict
ann_by = defaultdict(list)
for rec in rows:
    ann_by[rec['label']].append(rec)
deduped = []
dropped_within = 0
for lab, lst in ann_by.items():
    lst.sort(key=lambda x: (x['incident'], x['candidate']))
    kept = []
    for rec in lst:
        if any(iou(rec, k) >= DUP_IOU for k in kept):
            dropped_within += 1
        else:
            kept.append(rec)
    deduped.extend(kept)
print('after in-file dedup:', len(deduped), '(dropped', dropped_within, ')')

after in-file dedup: 200 (dropped 0 )


In [5]:
def load_existing(path):
    try:
        df = pd.read_csv(path)
        df['label'] = df['label'].astype(str).str.strip().str.lower()
        return df
    except Exception:
        return pd.DataFrame(columns=['incident', 'min_lat', 'max_lat', 'min_lon', 'max_lon'])

def dedupe_against(rows, existing):
    # Vectorized IoU sweep of candidate rows vs existing CSV rows; a row is kept
    # only if NO existing box overlaps it with IoU >= DUP_IOU.
    if existing.empty:
        return rows, 0
    E = existing[['min_lat', 'max_lat', 'min_lon', 'max_lon']].astype(float).to_numpy()
    e_lat  = (E[:, 0] + E[:, 1]) / 2.0
    e_hlat = (E[:, 1] - E[:, 0]) / 2.0
    e_lon  = (E[:, 2] + E[:, 3]) / 2.0
    e_hlon = (E[:, 3] - E[:, 2]) / 2.0
    area_e = (E[:, 1] - E[:, 0]) * (E[:, 3] - E[:, 2])
    kept, skipped, CH = [], 0, 256
    for start in range(0, len(rows), CH):
        chunk = rows[start:start + CH]
        R = np.array([[r['min_lat'], r['max_lat'], r['min_lon'], r['max_lon']]
                      for r in chunk], dtype=float)
        r_lat  = (R[:, 0] + R[:, 1]) / 2.0
        r_hlat = (R[:, 1] - R[:, 0]) / 2.0
        r_lon  = (R[:, 2] + R[:, 3]) / 2.0
        r_hlon = (R[:, 3] - R[:, 2]) / 2.0
        dlat  = np.abs(r_lat[:, None] - e_lat[None, :])
        dlon  = np.abs(r_lon[:, None] - e_lon[None, :])
        ov_lat = np.maximum(0.0, r_hlat[:, None] + e_hlat[None, :] - dlat)
        ov_lon = np.maximum(0.0, r_hlon[:, None] + e_hlon[None, :] - dlon)
        inter  = ov_lat * ov_lon
        area_r = (R[:, 1] - R[:, 0]) * (R[:, 3] - R[:, 2])
        union  = area_r[:, None] + area_e[None, :] - inter
        ious   = np.where(union > 0, inter / np.where(union > 0, union, 1.0), 0.0)
        mx = np.nanmax(ious, axis=1)
        for k, rec in enumerate(chunk):
            if mx[k] >= DUP_IOU:
                skipped += 1
            else:
                kept.append(rec)
    return kept, skipped

In [6]:
# 3) compare against existing CSVs; 4) keep only new rows; 5) append.
def append(path, rows):
    if rows:
        pd.DataFrame(rows).to_csv(path, index=False,
                                  header=not os.path.exists(path),
                                  mode='a')
    return len(rows)

ex_l = load_existing(LANDSLIDE_CSV)
ex_n = load_existing(NON_LANDSLIDES_CSV)

slide_rows,     skip_l = dedupe_against([r for r in deduped if r['label'] == 'landslide'], ex_l)
non_slide_rows, skip_n = dedupe_against([r for r in deduped if r['label'] == 'not_landslide'], ex_n)
print(f'  landslide:     {len(slide_rows) + skip_l} deduped -> {len(slide_rows)} new (skipped {skip_l} existing)')
print(f'  not_landslide: {len(non_slide_rows) + skip_n} deduped -> {len(non_slide_rows)} new (skipped {skip_n} existing)')

n_l = append(LANDSLIDE_CSV, slide_rows)
n_n = append(NON_LANDSLIDES_CSV, non_slide_rows)
print(f'appended {n_l} landslide + {n_n} non-landslide rows')

  landslide:     23 deduped -> 23 new (skipped 0 existing)
  not_landslide: 177 deduped -> 177 new (skipped 0 existing)
appended 23 landslide + 177 non-landslide rows
